# Data exploration and patient-level validation splits

This notebook prepares a stratified five-fold cross-validation split by patient for the binary task:

- `class_id` 1–3 → IDHmutant
- `class_id` 4–6 → IDHwildtype

The dataset contains sample entries, each with a 3D Raman array. A patient can have multiple entries, and each entry contains many spectra. The patient is therefore the split unit: every entry and spectrum from one patient must stay in the same fold.


In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import StratifiedKFold

PATH = r"..\data\preprocessed.npy"
dataset = np.load(PATH, allow_pickle=True)

print(f"Sample entries: {len(dataset)}")


Sample entries: 59


## 1. Build a sample-level metadata table

Each top-level dataset item is treated as one sample entry. The number of spectra is the product of the first two dimensions of `data` (for example, `(121, 51, 1167)` contains `121 × 51` spectra, each with 1167 channels). These counts are descriptive; they do not affect how patients are assigned to folds.


In [2]:
records = pd.DataFrame([
    {
        "entry_index": i,
        "patient_id": item["patient_id"],
        "sample_name": item["name"],
        "class_id": item["class_id"],
        "n_spectra": int(np.prod(item["data"].shape[:-1])),
        "n_channels": item["data"].shape[-1],
    }
    for i, item in enumerate(dataset)
])

records.head()


,entry_index,patient_id,sample_name,class_id,n_spectra,n_channels
0,0,305,HF-305_v4b_1_1.h5_6,2,6171,1167
1,1,442,HF-442_V4BB_1.h5_12,4,2500,1167
2,2,448,HF-448_V5B_1.h5_3,1,3600,1167
3,3,592,HF-592_V3T_1.h5_4,6,3600,1167
4,4,615,HF-615_V5BB_1.h5_9,2,3600,1167


## 2. Binary target mapping

Keep the original `class_id` in the table for traceability. `target` is the binary outcome used for stratification. The checks below catch unexpected class IDs and patients assigned to more than one class.


In [3]:
target_by_class = {
    1: "IDHmutant",
    2: "IDHmutant",
    3: "IDHmutant",
    4: "IDHwildtype",
    5: "IDHwildtype",
    6: "IDHwildtype",
}

records["target"] = records["class_id"].map(target_by_class)
assert records["target"].notna().all(), "Unexpected or missing class_id found."

class_counts_per_patient = records.groupby("patient_id")["class_id"].nunique()
assert class_counts_per_patient.eq(1).all(), "At least one patient has multiple class_id values."

records.head()


,entry_index,patient_id,sample_name,class_id,n_spectra,n_channels,target
0,0,305,HF-305_v4b_1_1.h5_6,2,6171,1167,IDHmutant
1,1,442,HF-442_V4BB_1.h5_12,4,2500,1167,IDHwildtype
2,2,448,HF-448_V5B_1.h5_3,1,3600,1167,IDHmutant
3,3,592,HF-592_V3T_1.h5_4,6,3600,1167,IDHwildtype
4,4,615,HF-615_V5BB_1.h5_9,2,3600,1167,IDHmutant


## 3. Summarize the cohort at the patient level

The patient table has one row per patient. It is the table used to create folds. Sample and spectrum counts are retained to make imbalance visible, but patients—not spectra—are stratified.


In [4]:
patients = (
    records.groupby("patient_id", as_index=False)
    .agg(
        target=("target", "first"),
        n_samples=("entry_index", "size"),
        n_spectra=("n_spectra", "sum"),
    )
)

cohort_summary = (
    patients.groupby("target")
    .agg(
        patients=("patient_id", "nunique"),
        sample_entries=("n_samples", "sum"),
        spectra=("n_spectra", "sum"),
    )
    .reindex(["IDHmutant", "IDHwildtype"])
)

print(f"Unique patients: {len(patients)}")
display(cohort_summary)


Unique patients: 46


,patients,sample_entries,spectra
target,,,
IDHmutant,21,26,167414
IDHwildtype,25,33,133092


## 4. Assign patients to five stratified folds

`StratifiedKFold` balances the number of **patients** in the two binary target groups as evenly as possible. The random seed makes the assignment reproducible. With only 46 patients, fold-level sample and spectrum totals can still differ because some patients contribute more data than others.


In [5]:
N_SPLITS = 5
RANDOM_STATE = 42

patients_per_class = patients["target"].value_counts()
assert patients_per_class.min() >= N_SPLITS, (
    "Five-fold stratification requires at least five patients in each target group."
)

splitter = StratifiedKFold(
    n_splits=N_SPLITS,
    shuffle=True,
    random_state=RANDOM_STATE,
)

patients["fold"] = -1
for fold, (_, validation_positions) in enumerate(
    splitter.split(patients["patient_id"], patients["target"])
):
    patients.loc[validation_positions, "fold"] = fold

assert patients["fold"].between(0, N_SPLITS - 1).all()
patients.sort_values(["fold", "target", "patient_id"]).reset_index(drop=True)


,patient_id,target,n_samples,n_spectra,fold
0,305,IDHmutant,1,6171,0
1,868,IDHmutant,1,7021,0
2,1016,IDHmutant,1,7497,0
3,1887,IDHmutant,1,5796,0
4,2534,IDHmutant,1,4096,0
5,1043,IDHwildtype,1,4096,0
6,2102,IDHwildtype,3,12288,0
7,2454,IDHwildtype,1,4096,0
8,2802,IDHwildtype,1,4096,0
9,2898,IDHwildtype,1,4096,0


## 5. Check fold balance and patient separation

This summary reports validation counts by fold. `validation_patients` and the class-specific patient counts describe the stratification; sample and spectrum counts show how much data each validation fold contains.


In [6]:
fold_rows = []
for fold in range(N_SPLITS):
    patient_fold = patients[patients["fold"] == fold]
    entry_fold = records[records["patient_id"].isin(patient_fold["patient_id"])]
    fold_rows.append({
        "fold": fold + 1,
        "validation_patients": len(patient_fold),
        "IDHmutant_patients": int((patient_fold["target"] == "IDHmutant").sum()),
        "IDHwildtype_patients": int((patient_fold["target"] == "IDHwildtype").sum()),
        "validation_sample_entries": len(entry_fold),
        "validation_spectra": int(entry_fold["n_spectra"].sum()),
    })

fold_summary = pd.DataFrame(fold_rows).set_index("fold")
display(fold_summary)


,validation_patients,IDHmutant_patients,IDHwildtype_patients,validation_sample_entries,validation_spectra
fold,,,,,
1,10,5,5,12,59253
2,9,4,5,11,54925
3,9,4,5,13,67396
4,9,4,5,11,57543
5,9,4,5,12,61389


In [7]:
# Verify every patient appears in exactly one validation fold.
assert patients["patient_id"].is_unique
assert patients["fold"].notna().all()
assert patients.groupby("patient_id")["fold"].nunique().eq(1).all()

# Verify the patient sets used for training and validation are disjoint in every fold.
for fold in range(N_SPLITS):
    validation_patients = set(patients.loc[patients["fold"] == fold, "patient_id"])
    training_patients = set(patients.loc[patients["fold"] != fold, "patient_id"])
    assert training_patients.isdisjoint(validation_patients)
    print(
        f"Fold {fold + 1}: {len(training_patients)} training patients, "
        f"{len(validation_patients)} validation patients; no patient overlap"
    )


Fold 1: 36 training patients, 10 validation patients; no patient overlap
Fold 2: 37 training patients, 9 validation patients; no patient overlap
Fold 3: 37 training patients, 9 validation patients; no patient overlap
Fold 4: 37 training patients, 9 validation patients; no patient overlap
Fold 5: 37 training patients, 9 validation patients; no patient overlap


## 6. Map patient folds back to sample-entry indices

Use these entry indices to select whole sample records for a fold. When spectra are extracted from an entry, they inherit that entry's patient fold. Do not randomly split individual spectra or pixels across training and validation.


In [8]:
fold_by_patient = patients.set_index("patient_id")["fold"]
records["fold"] = records["patient_id"].map(fold_by_patient)

fold_splits = []
for fold in range(N_SPLITS):
    validation_entries = records.loc[records["fold"] == fold, "entry_index"].to_numpy()
    training_entries = records.loc[records["fold"] != fold, "entry_index"].to_numpy()

    training_patients = set(records.loc[records["fold"] != fold, "patient_id"])
    validation_patients = set(records.loc[records["fold"] == fold, "patient_id"])
    assert training_patients.isdisjoint(validation_patients)

    fold_splits.append({
        "fold": fold + 1,
        "train_entry_indices": training_entries,
        "validation_entry_indices": validation_entries,
        "train_patient_ids": training_patients,
        "validation_patient_ids": validation_patients,
    })

# Example: sample-entry indices for the first fold
print("Training entries:", fold_splits[0]["train_entry_indices"])
print("Validation entries:", fold_splits[0]["validation_entry_indices"])


Training entries: [ 1  2  3  4  5  6  8  9 10 11 12 13 14 15 16 19 20 21 22 23 25 29 30 31
 32 33 35 36 37 39 40 41 42 43 44 45 46 47 48 50 51 52 53 54 55 57 58]
Validation entries: [ 0  7 17 18 24 26 27 28 34 38 49 56]


## Notes for the modeling notebook

- The fold assignment is at the patient level, even when fitting a model on spectra.
- Decide before evaluation how spectrum-level predictions will be combined into sample- or patient-level predictions. Report the evaluation unit clearly; thousands of spectra are not thousands of independent patients.
- If samples with more spectra should not dominate training or metrics, use a pre-specified sample/patient weighting or aggregation strategy.
- Fit preprocessing, feature selection, and any data-derived dimensionality reduction using training data inside each fold. Do not use precomputed `pca` or `tsne` coordinates as predictors for validation unless they are recomputed without validation-data leakage.
- If model or kernel settings are tuned, use tuning confined to each training fold (for example, inner cross-validation); keep the outer five folds for evaluation.
